In [ ]:
import sys

sys.path.append("..")
import os
import time
import glob
import numpy as np
import pandas as pd
from typing import List, Dict, Union, Tuple
from utils.data import load_yaml
from imaris.imaris import ImarisDataObject
from parsers.spot_track_parser import SpotTrackParserDistributed
from parsers.spot_track_object_parser import SpotTrackObjectParserDistributed
from tqdm.notebook import tqdm

import csv

Get Stats From Parser

In [ ]:
# test ims file path
test_ims_file = "../../data/imaris_parser_test_files/Spots/KO Sec1 Roi1 3x3 80min.ims"

In [ ]:
parser = SpotTrackParserDistributed(test_ims_file)
out = parser.inspect(1)
parser_df = out["final_df"]
parser_stat_names = list(parser_df.columns)

Get Stats From Test Files

In [ ]:
# track stats folder
track_stats_dir = "../../data/imaris_parser_test_files/Spots/KO Sec1 Roi1 3x3 80min-testtracks_Statistics"

# object stats folder
# object_stats_dir = (
#     "../../data/imaris_parser_test_files/Spots/KO Sec1 Roi1 3x3 80min_Statistics"
# )

In [ ]:
track_stats_files = glob.glob(os.path.join(track_stats_dir, "*.csv"))

In [ ]:
def get_stats_df(path: str) -> Tuple[str, pd.DataFrame]:
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        dataframe = []
        for row in reader:
            dataframe.append(row)

    # get the stats name
    stat_name = dataframe[1][0]

    # the first three rows from imaris export needs to be dropped
    dataframe = pd.DataFrame(dataframe[4:], columns=dataframe[3])
    # out = {stat_name: dataframe}
    return (stat_name, dataframe)


index = 6
items = get_stats_df(track_stats_files[index])
print(items[0], "\n")
items[1]

In [ ]:
# get all the stat from the test files into a dict
test_stats_dict = {}
for stat_test_path in tqdm(track_stats_files):
    items = get_stats_df(stat_test_path)
    if items[0] in test_stats_dict.keys():
        raise ValueError
    else:
        test_stats_dict[items[0]] = items[1]

In [ ]:
print(f"Num stats to evaluate: {len(parser_stat_names)}")

for stat_name in tqdm(parser_stat_names):
    # get test stats df
    if stat_name != "Track_ID":

        test_df = test_stats_dict[stat_name]

        # it seems like if channel and image information is given the df will modify its name
        # we need to drop these
        stat_name_our_df = None
        if "Channel" in list(test_df.columns) and "Image" in list(test_df.columns):
            index = stat_name.find("Ch=") - 1
            stat_name_our_df = stat_name[:index]

        if "Spots" in list(test_df.columns):
            if stat_name_our_df:
                index = stat_name_our_df.find("Spots=") - 1
                stat_name_our_df = stat_name_our_df[:index]
            else:
                index = stat_name.find("Spots=") - 1
                stat_name_our_df = stat_name[:index]

        # get df to compare to
        if stat_name_our_df:
            test_df = test_stats_dict[stat_name].filter([stat_name_our_df, "ID"])
        else:
            test_df = test_stats_dict[stat_name].filter([stat_name, "ID"])

        # get the df we generate
        our_df = parser_df.filter([stat_name, "Track_ID"]).rename(
            columns={"Track_ID": "ID"}
        )
        our_df = our_df.reset_index().drop("index", axis=1)
        our_df["ID"] = our_df["ID"].astype(str)
        our_df[stat_name] = our_df[stat_name].astype(float).map(lambda x: f"{x:.3f}")
        our_df[stat_name] = our_df[stat_name].astype(str)

        # it seems like if channel and image information is given the df will modify its name
        # we need to drop these
        if stat_name_our_df:
            our_df = our_df.rename(columns={stat_name: stat_name_our_df})

        if not (test_df.equals(our_df)):
            print(stat_name)
            print(stat_name_our_df)
            break

In [ ]:
our_df

In [ ]:
test_df

In [ ]:
t = test_stats_dict["Track Intensity Center Mean Ch=1 Img=1"]
o = parser_df.filter(["Track Intensity Sum Ch=3 Img=1", "Track_ID"]).rename(
    columns={"Track_ID": "ID"}
)
t

In [ ]:
stat_name = "Track Intensity Center Mean"

In [ ]:
if "Channel" in list(t.columns) and "Image" in list(t.columns):
    stat_name_new = (
        f"{stat_name} Ch={t.iloc[0]['Channel']} Img={t.iloc[0]['Image'].split(" ")[-1]}"
    )
    t = t.rename(columns={stat_name: stat_name_new})

In [ ]:
t

In [ ]:
stat_name = f"{stat_name} Ch={t.iloc[0]}"

In [ ]:
list(test_stats_dict.keys())

In [ ]:
test_stats_dict["Intensity Sum of Square Ch=2 Img=1"]

In [ ]:
list(parser_df.columns)

In [ ]:
len(test_stats_dict.keys())